# 机械臂标定

## 导入依赖

In [1]:
import math
from kyle_robot_toolbox.robot_arm.arm4dof_uservo import Arm4DoFUServo

## 机械臂初始化

In [2]:
# 创建机械臂
arm = Arm4DoFUServo(config_folder="./config", \
                    is_init_pose=False)

## 设置舵机为阻尼模式

In [3]:
# 设置为阻尼模式
# 数值越大，阻尼越大
arm.set_damping(1000) 

## 样本采集

将机械臂掰直, 水平于桌面, 夹爪张开。 采集舵机原始角度

![](image/pose1.jpg)

In [4]:
pose1_servo_raw_angle = arm.get_servo_angle_list()
# 打印舵机原始角度
j1_0, j2_n90, j3_90, j4_n90, j5_0, = pose1_servo_raw_angle
print(f"J1: {j1_0:.1f} J2: {j2_n90:.1f} J3: {j3_90:.1f} J4: {j4_n90:.1f} J5: {j5_0:.1f}")    

J1: -1.2 J2: 3.7 J3: -89.9 J4: 88.6 J5: 94.3


夹爪闭合, 令夹爪运动到机械臂左侧，夹爪末端垂直于桌面，且连杆之间均为90度直角。

![](./image/pose2.jpg)

In [5]:
pose2_servo_raw_angle = arm.get_servo_angle_list()
# 打印舵机原始角度
j1_90, j2_0, j3_0, j4_0, j5_90 = pose2_servo_raw_angle
print(f"J1: {j1_90:.1f} J2: {j2_0:.1f} J3: {j3_0:.1f} J4: {j4_0:.1f} J5: {j5_90:.1f}")    

J1: -92.9 J2: 91.6 J3: 0.0 J4: 0.8 J5: 1.7


## 标定舵机

计算各个关节的比例系数与角度偏移量. 用一个简单的一次函数来表示舵机原始角度与机械臂关节弧度之间的关系

$$
angle_i = k_i*\theta_{i} + b_i
$$

* $\theta_i$关节弧度
* $angle_i$ 舵机原始角度
* $k_i$ 比例系数
* $b_i$ 偏移量

In [6]:
def calc_kb(angle_a, angle_b, theta_a, theta_b):
    k = (angle_a-angle_b) / (theta_a-theta_b)
    b = angle_a - k*theta_a
    return k, b

In [7]:
k1, b1 = calc_kb(j1_0, j1_90, 0, math.pi/2)
k2, b2 = calc_kb(j2_n90, j2_0, -math.pi/2, 0)
k3, b3 = calc_kb(j3_90, j3_0, math.pi/2, 0)
k4, b4 = calc_kb(j4_n90, j4_0, -math.pi/2, 0)
k5, b5 = calc_kb(j5_0, j5_90, math.pi/2, 0)

print("joint2servo:")
print(f'  k: [{k1:.3f}, {k2:.3f}, {k3:.3f}, {k4:.3f}, {k5:.3f}]')
print(f"  b: [{b1:.3f}, {b2:.3f}, {b3:.3f}, {b4:.3f}, {b5:.3f}]")

joint2servo:
  k: [-58.378, 55.959, -57.232, -55.895, 58.951]
  b: [-1.200, 91.600, 0.000, 0.800, 1.700]


将打印出来的字符串替换掉`config/arm.yaml`里面的对应配置项。 